# Bagheria 2026 — analisi condivisa

Base comune ai tre thread esplorativi. Carica `data/processed/`, verifica che le definizioni
fissate per tutto il progetto (relazione, `dist/RELAZIONE_DATAPOLIS.pdf` §1.1) reggano sui dati reali, e produce i numeri di riferimento che gli altri
notebook danno per calcolati: popolazione giovane (2021-2024), benchmark 2011, condizione
professionale 15-24 (2018-2024, senza il 2020).

**Prerequisito**: `uv run python -m pipeline.build`. I dati grezzi sono già in `data/raw/`;
`pipeline.fetch` serve solo ad aggiornarli, e scarica file nuovi datati al giorno del download.
Questo notebook non scarica niente e non tocca `data/raw/`: legge solo `data/processed/`.

**Dettaglio delle fonti, endpoint e trappole**: `docs/sources.md`.

# **Caricamento**
Le sei tabelle prodotte da `pipeline/build.py` e i quattro territori di confronto.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

RADICE = Path.cwd()
if RADICE.name == "notebooks":   # nbconvert gira dentro notebooks/, l'editor spesso dalla radice
    RADICE = RADICE.parent
PROCESSED = RADICE / "data" / "processed"


def leggi(nome: str) -> pd.DataFrame:
    """Legge una tabella di data/processed tenendo i codici come stringhe.

    Non è pignoleria: `condizione` contiene 1, 12, 99 e pandas li indovinerebbe come
    interi, facendo fallire in silenzio ogni filtro scritto come stringa.
    """
    tabella = pd.read_csv(PROCESSED / nome, dtype=str)
    for colonna in ("valore", "eta_anni"):
        if colonna in tabella.columns:
            tabella[colonna] = pd.to_numeric(tabella[colonna])
    if "anno" in tabella.columns:
        tabella["anno"] = tabella["anno"].astype(int)
    return tabella


territori = leggi("territori.csv")
indicatori = leggi("indicatori.csv")
codici = leggi("codici.csv")
ottomila = leggi("ottomilacensus_long.csv")
istr_lav = leggi("censpop_istr_lav_long.csv")
popolazione = leggi("censpop_popolazione_long.csv")

# I quattro territori di confronto fissati per tutto il progetto.
BAGHERIA, PALERMO, SICILIA, ITALIA = "082006", "082053", "ITG1", "IT"
CONFRONTO = [BAGHERIA, PALERMO, SICILIA, ITALIA]
NOMI = territori.set_index("territorio")["nome_territorio"].to_dict()

pd.DataFrame(
    [(nome, len(t), t.shape[1]) for nome, t in [
        ("territori", territori), ("indicatori", indicatori), ("codici", codici),
        ("ottomilacensus_long", ottomila), ("censpop_istr_lav_long", istr_lav),
        ("censpop_popolazione_long", popolazione)]],
    columns=["tabella", "righe", "colonne"])

,tabella,righe,colonne
0,territori,521,4
1,indicatori,99,4
2,codici,164,3
3,ottomilacensus_long,154737,6
4,censpop_istr_lav_long,6552,10
5,censpop_popolazione_long,14462,8


# **Verifica delle definizioni fissate**
Prima di analizzare: i quattro territori esistono davvero in entrambe le fonti, e quali
fasce d'età sono realmente disponibili. È la cella che giustifica le definizioni del progetto.

In [2]:
# 1. I quattro territori di confronto esistono in entrambe le fonti?
presenza = pd.DataFrame({
    "nome": [NOMI[c] for c in CONFRONTO],
    "8milacensus": [c in set(ottomila["territorio"]) for c in CONFRONTO],
    "censpop": [c in set(istr_lav["territorio"]) for c in CONFRONTO],
}, index=CONFRONTO)
print("Territori di confronto:", "\n", presenza, "\n")

# 2. Che anni copre ciascuna fonte? Non basta guardare la tabella nel suo insieme:
#    la copertura cambia a seconda della fascia d'età, ed è quello che determina le serie.
anni = lambda t: sorted(t["anno"].unique().tolist())
eta_singole = popolazione[popolazione["eta_anni"].notna() & popolazione["cittadinanza"].eq("TOTAL")]
giovani_15_24 = istr_lav[istr_lav["tavola"].eq("lavoro") & istr_lav["eta"].eq("Y15-24")]

print("8milaCensus, anni censuari:            ", anni(ottomila))
print("Censimento permanente, tabella intera: ", anni(istr_lav))
print("  di cui classe 15-24 (lavoro):        ", anni(giovani_15_24))
print("Popolazione, tabella intera:           ", anni(popolazione))
print("  di cui età singole (cittadinanza TOTAL):", anni(eta_singole), "\n")

# 3. Quali classi d'età esistono su lavoro e istruzione? È il vincolo che decide il taglio.
etichette_eta = codici[codici["dimensione"].eq("eta")][["codice", "etichetta"]]
(istr_lav[["tavola", "eta"]].drop_duplicates()
    .merge(etichette_eta, left_on="eta", right_on="codice", how="left")
    .pivot_table(index=["eta", "etichetta"], columns="tavola", values="codice", aggfunc="size", fill_value=0))

Territori di confronto: 
             nome  8milacensus  censpop
082006  Bagheria         True     True
082053   Palermo         True     True
ITG1     Sicilia         True     True
IT        Italia         True     True 

8milaCensus, anni censuari:             [1991, 2001, 2011]
Censimento permanente, tabella intera:  [2018, 2019, 2020, 2021, 2022, 2023, 2024]
  di cui classe 15-24 (lavoro):         [2018, 2019, 2021, 2022, 2023, 2024]
Popolazione, tabella intera:            [2018, 2019, 2020, 2021, 2022, 2023, 2024]
  di cui età singole (cittadinanza TOTAL): [2021, 2022, 2023, 2024] 



,tavola,istruzione,lavoro
eta,etichetta,,
Y15-24,15-24 anni,0,1
Y25-49,25-49 anni,1,1
Y50-64,50-64 anni,1,1
Y9-24,9-24 anni,1,0
Y_GE15,15 anni e più,0,1
Y_GE65,65 anni e più,1,1
Y_GE9,9 anni e più,1,0


> I quattro territori ci sono in entrambe le fonti, quindi il confronto regge senza mappature.
>
> Le classi d'età confermano il vincolo: su lavoro e istruzione **non esiste nessuna fascia che
> arrivi a 34 anni** — si passa da `Y15-24` a `Y25-49`. La fascia 15-34 del bando è ricostruibile
> solo in demografia, dove ci sono le età singole. Per questo il progetto fissa 15-24 su
> lavoro/istruzione e 15-34 solo sulla popolazione.
>
> Due buchi di copertura da tenere presenti, ed è il motivo per cui la verifica guarda le fasce
> d'età una per una invece della tabella nel suo insieme:
>
> - **Il 2020 manca sulla classe 15-24**, in tutti e quattro i territori. La tabella contiene il
>   2020 per altre fasce, quindi un controllo sulla tabella intera non lo avrebbe visto.
> - **Le età singole con cittadinanza `TOTAL` esistono solo dal 2021**, benché le classi aggregate
>   partano dal 2018. La serie 15-34 è quindi 2021-2024, non 2018-2024.
>
> Le celle successive usano gli anni effettivamente disponibili. Non si interpola e non si
> riempiono i buchi: sono dati mancanti alla fonte, non rumore da lisciare.

# **Popolazione 15-34**
Il target del bando, ricostruito sommando le età singole. È l'unico posto dove la fascia
15-34 è esatta. Misura lo **stock**, non l'emigrazione: la variazione somma il ricambio
delle coorti (chi compie 15 anni contro chi supera i 34) e il saldo migratorio. La scomposizione
sta in `genere.ipynb` («Lo stock non è la fuga»); per la fuga si usa la ritenzione di coorte.

In [3]:
GIOVANI = (15, 34)

giovani = (popolazione[
        popolazione["territorio"].isin(CONFRONTO)
        & popolazione["cittadinanza"].eq("TOTAL")
        & popolazione["eta_anni"].between(*GIOVANI)]
    .groupby(["territorio", "anno", "genere"], as_index=False)["valore"].sum()
    .rename(columns={"valore": "popolazione_15_34"}))
giovani["nome_territorio"] = giovani["territorio"].map(NOMI)
giovani.to_csv(PROCESSED / "analisi_popolazione_giovane.csv", index=False)

# Serie di Bagheria per genere, con la variazione cumulata rispetto al primo anno disponibile.
serie = (giovani[giovani["territorio"].eq(BAGHERIA)]
         .pivot(index="anno", columns="genere", values="popolazione_15_34")
         .rename(columns={"M": "maschi", "F": "femmine", "T": "totale"}))
serie["var. % su primo anno"] = (100 * serie["totale"] / serie["totale"].iloc[0] - 100).round(1)
serie

genere,femmine,maschi,totale,var. % su primo anno
anno,,,,
2021,6054,6120,12174,0.0
2022,6010,6108,12118,-0.5
2023,5955,6032,11987,-1.5
2024,5846,6015,11861,-2.6


In [4]:
# Lo stesso calo, confrontato con i territori di riferimento: in punti percentuali,
# non in valori assoluti, altrimenti Bagheria e Italia non sono paragonabili.
confronto_calo = (giovani[giovani["genere"].eq("T")]
    .pivot(index="anno", columns="nome_territorio", values="popolazione_15_34"))
(100 * confronto_calo / confronto_calo.iloc[0] - 100).round(1)

nome_territorio,Bagheria,Italia,Palermo,Sicilia
anno,,,,
2021,0.0,0.0,0.0,0.0
2022,-0.5,0.5,-0.8,-0.9
2023,-1.5,0.8,-1.5,-1.9
2024,-2.6,1.2,-1.9,-2.6


> La lettura va fatta sulla seconda tabella, non sulla prima: in valori assoluti un comune e
> l'Italia non si confrontano. Fra 2021 e 2024 lo stock 15-34 di Bagheria cala del 2,6%, come
> quello della Sicilia e più di Palermo (−1,9%), mentre l'Italia cresce (+1,2%) per
> immigrazione. È una variazione di stock: dice che la platea giovane si restringe, non da sola
> quanti giovani partono.

# **Benchmark 2011 — indicatori giovanili**
Gli indicatori 8milaCensus sui giovani, con il gap di Bagheria verso ciascun territorio.
Sono dati **2011**: fotografia storica, non lo stato attuale.

In [5]:
INDICATORI_GIOVANI = ["L4", "V8", "L14", "L9", "L13", "I7", "I5", "I8"]

bench = (ottomila[
        ottomila["territorio"].isin(CONFRONTO)
        & ottomila["anno"].eq(2011)
        & ottomila["indicatore"].isin(INDICATORI_GIOVANI)]
    .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore")
    .pivot(index=["indicatore", "nome_indicatore"], columns="territorio", values="valore"))
bench = bench[CONFRONTO].rename(columns={c: NOMI[c] for c in CONFRONTO})

# Regola del progetto: sempre valore assoluto e gap, mai solo assoluti.
for altro in ["Palermo", "Sicilia", "Italia"]:
    bench[f"gap vs {altro}"] = (bench["Bagheria"] - bench[altro]).round(1)

bench.round(1)

,territorio,Bagheria,Palermo,Sicilia,Italia,gap vs Palermo,gap vs Sicilia,gap vs Italia
indicatore,nome_indicatore,,,,,,,
I5,Uscita precoce dal sistema di istruzione e formazione,28.6,25.8,23.2,15.5,2.8,5.4,13.1
I7,Incidenza di giovani con istruzione universitaria,14.4,20.6,18.3,23.2,-6.2,-3.9,-8.8
I8,Livello di istruzione dei giovani 15-19 anni,96.7,95.6,96.5,97.9,1.1,0.2,-1.2
L13,Indice di ricambio occupazionale,268.8,363.0,295.3,298.1,-94.2,-26.5,-29.3
L14,Tasso di occupazione 15-29 anni,20.0,20.6,24.3,36.3,-0.6,-4.3,-16.3
L4,Incidenza giovani 15-29 anni che non studiano e non lavorano,40.1,38.8,34.7,22.5,1.3,5.4,17.6
L9,Tasso di disoccupazione giovanile,62.4,64.4,53.7,34.7,-2.0,8.7,27.7
V8,Incidenza di giovani fuori dal mercato del lavoro e dalla formazione,23.4,19.9,19.4,12.3,3.5,4.0,11.1


> `L4` (NEET 15-29) e `L14` (occupazione 15-29) sono le due righe più vicine al target del
> bando, e vanno citate **sempre con l'anno**: sono 2011. Il censimento permanente non le ricostruisce, perché
> la fascia 15-29 a livello comunale non esiste più — vedi la verifica delle classi d'età sopra.

# **Condizione professionale 15-24, 2018-2024**
Il dato attuale, per genere. Tre misure, con definizioni fissate qui una volta per tutte
così che i tre thread non ne inventino tre versioni diverse.

- **tasso di occupazione** = occupati / popolazione;
- **tasso di disoccupazione** = in cerca / forze di lavoro;
- **fuori da lavoro e studio** = (popolazione − occupati − studenti) / popolazione, cioè in cerca +
  casalinghe/i + pensione + altra condizione. È un proxy su 15-24, **non** il NEET ISTAT 15-29, e
  non va mai messo in serie con `L4` (2011). La sua parte che non cerca lavoro sono gli
  «inattivi non studenti» del notebook educazione.

**Rottura di misura 2019→2021**: la quota «in cerca» cade in un solo passaggio in tutti i
territori (cambio del metodo di stima del censimento permanente, `docs/sources.md` §7).
Disoccupazione e «fuori da lavoro e studio» si confrontano fra territori nello stesso anno, non
lungo la serie a cavallo del 2020; il tasso di occupazione non ne è toccato.

In [6]:
CLASSE_GIOVANE = "Y15-24"

base = istr_lav[
    istr_lav["tavola"].eq("lavoro")
    & istr_lav["territorio"].isin(CONFRONTO)
    & istr_lav["eta"].eq(CLASSE_GIOVANE)
    & istr_lav["cittadinanza"].eq("TOTAL")
    & istr_lav["titolo_studio"].eq("ALL")
    & istr_lav["genere"].isin(["M", "F", "T"])]

# Codici CL_FORZE_LAV: 99 totale, 1 occupato, 12 in cerca, 22 forze di lavoro, 5 studente.
conteggi = base.pivot_table(index=["territorio", "anno", "genere"],
                            columns="condizione", values="valore", aggfunc="sum")

tassi = pd.DataFrame({
    "popolazione": conteggi["99"],
    "occupati": conteggi["1"],
    # Occupati sulla popolazione: stessa definizione di L14 in 8milaCensus, fascia diversa.
    "tasso_occupazione": 100 * conteggi["1"] / conteggi["99"],
    # In cerca sulle forze di lavoro, non sulla popolazione.
    "tasso_disoccupazione": 100 * conteggi["12"] / conteggi["22"],
    # Chi non lavora e non studia. NON è il NEET ISTAT 15-29: fascia diversa e definizione
    # diversa, si veda docs/sources.md. Va sempre etichettato "15-24".
    "fuori_da_lavoro_e_studio": 100 * (conteggi["99"] - conteggi["1"] - conteggi["5"]) / conteggi["99"],
}).round(1).reset_index()
tassi["nome_territorio"] = tassi["territorio"].map(NOMI)
tassi.to_csv(PROCESSED / "analisi_condizione_15_24.csv", index=False)

tassi[tassi["territorio"].eq(BAGHERIA) & tassi["genere"].eq("T")].set_index("anno")[
    ["popolazione", "occupati", "tasso_occupazione", "tasso_disoccupazione", "fuori_da_lavoro_e_studio"]]

,popolazione,occupati,tasso_occupazione,tasso_disoccupazione,fuori_da_lavoro_e_studio
anno,,,,,
2018,6243.0,512.0,8.2,68.8,37.7
2019,6134.0,527.0,8.6,67.9,32.8
2021,5891.0,595.0,10.1,51.3,29.8
2022,5882.0,686.0,11.7,46.5,27.3
2023,5879.0,686.0,11.7,49.5,30.7
2024,5904.0,734.0,12.4,38.9,26.9


In [7]:
# Gap di genere sul tasso di occupazione 15-24: quadro di sintesi per i quattro territori.
# La decomposizione per titolo di studio non è calcolabile a livello comunale: vedi
# genere.ipynb, «Verifica di fattibilità».
gap = (tassi[tassi["genere"].isin(["M", "F"])]
       .pivot_table(index=["nome_territorio", "anno"], columns="genere", values="tasso_occupazione"))
gap["gap M-F (punti)"] = (gap["M"] - gap["F"]).round(1)
gap.loc[gap.index.get_level_values("anno").isin([2021, 2024])].round(1)

genere                   F     M  gap M-F (punti)
nome_territorio anno                             
Bagheria        2021   6.2  13.8              7.6
                2024   8.2  16.5              8.3
Italia          2021  15.0  24.8              9.8
                2024  17.3  26.9              9.6
Palermo         2021   7.5  13.9              6.4
                2024   9.6  16.5              6.9
Sicilia         2021   8.4  17.2              8.8
                2024  10.4  20.3              9.9

> Il gap di genere sull'occupazione giovanile è la misura che il thread Genere approfondisce.
> Qui serve solo come quadro condiviso: nel 2024 il gap di Bagheria (8,3 punti) è sopra
> Palermo (6,9) e sotto Sicilia (9,9) e Italia (9,6), quindi riproduce un divario regionale;
> lo specifico di Bagheria è il livello del tasso femminile, il più basso dei quattro.
>
> Attenzione a leggere `tasso_disoccupazione` e `fuori_da_lavoro_e_studio` insieme: il primo ha
> per denominatore le forze di lavoro, il secondo la popolazione. Su una fascia dove molti
> studiano ancora, il secondo è il più informativo.

# **Tabelle esportate per le figure R**
Gli script in `viz/` leggono questi file. La logica di trasformazione resta qui, in Python:
in R si filtra e si disegna, non si ricalcola.

In [8]:
pd.DataFrame(
    [(p.name, sum(1 for _ in p.open()) - 1) for p in sorted(PROCESSED.glob("analisi_*.csv"))],
    columns=["file", "righe"])

,file,righe
0,analisi_condizione_15_24.csv,72
1,analisi_popolazione_giovane.csv,48
